# Customer 360 -- Fusing the Data Warehouse with the Documents Customers Generate

**The signals that predict churn live in two places that rarely meet.** Structured facts sit in the warehouse -- usage telemetry, survey scores, transactions, campaigns. The *why* sits in unstructured docs -- the support ticket, the angry chat, the survey free-text. This pipeline reconciles both into one per-customer record.

It:

1. **Classifies** every customer document by type -- the *doc gate* (`AI_CLASSIFY`); junk drops to `other`.
2. **Extracts a signal** (sentiment + issue) from each doc with `AI_COMPLETE`.
3. **Fuses** those AI signals with six structured tables in one warehouse `JOIN` -> a `RISK_TIER` and a `ROUTE` for *every* customer.
4. **Searches** the doc text (Cortex Search) so you can pull the exact evidence behind a tier.
5. **Summarizes** product health with an AI-written executive briefing per product.

Everything is declarative Snowflake Cortex SQL on incremental dynamic tables: each AI function runs **once per new document**, not once per refresh.

```
DEMO_C360_* structured tables (pre-loaded)      DEMO_C360_DOCS_STAGE (customer docs in incoming/)
                                                  -> DEMO_C360_FILE_LOG        stream + task (event-driven ingest)
                                                  -> DT_DEMO_C360_CLASSIFIED   AI_CLASSIFY -> DOC_TYPE     [doc gate]
                                                  -> DT_DEMO_C360_CUSTOMER_DOCS attach CONTENT, drop 'other'
                                                  -> DT_DEMO_C360_DOC_SIGNALS   AI_COMPLETE(json)  sentiment + issue
                                                       |
  the six tables + the doc signals  ---- JOIN ---->  DT_DEMO_C360_CUSTOMER_RECORD   RISK_TIER + ROUTE   [deliverable]
                                                  -> DT_DEMO_C360_SEARCH_CHUNKS + DEMO_C360_SEARCH   Cortex Search
                                                  -> DT_DEMO_C360_HEALTH_LANDSCAPE  AI_COMPLETE briefing per product [deliverable]
                                                  -> views DEMO_C360_HIGH_RISK / _NEEDS_REVIEW / _AUTO_ACT / _CUSTOMER_360
```

`CUSTOMER_ID` is parsed from the staged path (`incoming/<customer_id>__<type>.txt`) to link a doc to its account; classification itself is honest AI over the document text. Each customer carries a planted `COHORT_STORY` that fusion uses only as a guardrail (and that we grade against at the end).

> **Before running:** this notebook reads objects the pipeline already built, so first run `00_setup.sql`, the sourcing script (`source_customer360.py`, which loads the six structured tables and backfills the file log + `CONTENT`), `10_pipeline.sql` (creates the dynamic tables, zero-spend), and `20_insights.sql` section A (the cost-gated AI refresh) -- then let the dynamic tables settle. Substitute `{database}` / `{schema}` / `{warehouse}` in the context cell below; all other object references resolve against the schema it sets.

In [ ]:
USE SCHEMA {database}.{schema};
USE WAREHOUSE {warehouse};

## 1 - The intake -- what arrived, and what each document is

The first job is the *doc gate*: one `AI_CLASSIFY` call per file. Note the `other` bucket -- non-customer "junk" documents (an internal newsletter, a vendor invoice) are gated out here, so they never contribute a signal to a customer.

In [ ]:
SELECT DOC_TYPE,
       COUNT(*)                                           AS documents,
       ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (), 1)  AS pct
FROM DT_DEMO_C360_CLASSIFIED
GROUP BY DOC_TYPE
ORDER BY documents DESC;

Every dynamic table is **incremental** -- its `refresh_mode` is `INCREMENTAL`, so when new documents land only those documents are processed; the AI functions never re-run on unchanged files. This is what makes the same SQL that runs on ~40 customers run on millions.

In [ ]:
SHOW DYNAMIC TABLES LIKE 'DT_DEMO_C360%';
SELECT "name", "refresh_mode", "target_lag", "scheduling_state"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()))
ORDER BY "name";

## 2 - Fusion -- one record per customer

This is the hero. The six structured tables are `LEFT JOIN`ed to the AI doc signals into a single row per customer. `MAX_ERROR_RATE` and `DAU_DECLINE_PCT` come from telemetry, `NPS_Q2` from surveys, `TOTAL_CHARGED` from transactions, `PRODUCT_CATEGORY` from the product dimension, `CAMPAIGN_EXPOSED` from campaigns -- and `NEG_DOC_COUNT` is the count of documents the model read as **negative**. `RISK_TIER` and `ROUTE` are then decided by SQL rules over all of it. Every customer is scored, whether or not they ever filed a document.

In [ ]:
SELECT CUSTOMER_ID, COMPANY_NAME, PRIMARY_PRODUCT, PRODUCT_CATEGORY,
       NPS_Q2,
       ROUND(MAX_ERROR_RATE, 3)  AS max_error_rate,
       ROUND(DAU_DECLINE_PCT, 2) AS dau_decline_pct,
       DOC_COUNT, NEG_DOC_COUNT,
       RISK_TIER, ROUTE
FROM DEMO_C360_CUSTOMER_360
ORDER BY CUSTOMER_ID
LIMIT 12;

## 3 - Risk tier x route -- the portfolio at a glance

The distribution across *all* customers. `escalate` (a live error spike), `needs_review` (a soft signal a human should look at), `auto_act` (healthy). This is the routing an ops team acts on each morning.

In [ ]:
SELECT RISK_TIER, ROUTE,
       COUNT(*)                                           AS customers,
       ROUND(100 * RATIO_TO_REPORT(COUNT(*)) OVER (), 1)  AS pct
FROM DT_DEMO_C360_CUSTOMER_RECORD
GROUP BY RISK_TIER, ROUTE
ORDER BY customers DESC;

## 4 - The fusion proof -- risk that only the documents reveal

These customers look **fine** on *every* structured signal -- no error spike, no usage cliff, and a healthy NPS above 8 -- yet they are flagged risk purely because their documents read negative. The `NPS_Q2 > 8` filter rules out survey-driven risk, so the negative document is the *only* thing lifting them above `low`. A structured-only dashboard would show them as green. This is the row fusion exists to surface.

In [ ]:
SELECT CUSTOMER_ID, COMPANY_NAME, PRIMARY_PRODUCT, RISK_TIER, NPS_Q2,
       ROUND(MAX_ERROR_RATE, 3) AS max_error_rate,
       NEG_DOC_COUNT
FROM DT_DEMO_C360_CUSTOMER_RECORD
WHERE NEG_DOC_COUNT >= 1
  AND COALESCE(MAX_ERROR_RATE, 0) <= 0.05
  AND COALESCE(DAU_DECLINE_PCT, 0) > -0.15
  AND COALESCE(NPS_Q2, 10) > 8
  AND RISK_TIER <> 'low'
ORDER BY NEG_DOC_COUNT DESC, CUSTOMER_ID
LIMIT 15;

## 5 - The high-risk queue -- who a CSM calls first

The high-risk accounts, largest by revenue first, with the fused evidence side by side so the reason for the tier is legible: the structured facts *and* the negative-doc count that AI contributed.

In [ ]:
SELECT CUSTOMER_ID, COMPANY_NAME, PRIMARY_PRODUCT,
       ROUND(TOTAL_CHARGED)      AS total_charged,
       NPS_Q2,
       ROUND(MAX_ERROR_RATE, 3)  AS max_error_rate,
       ROUND(DAU_DECLINE_PCT, 2) AS dau_decline_pct,
       NEG_DOC_COUNT, ROUTE
FROM DEMO_C360_HIGH_RISK
LIMIT 15;

## 6 - Pull the evidence -- Cortex Search over the docs

Fusion gives you the *score*; Cortex Search gives you the *quote*. The same doc text that drove `NEG_DOC_COUNT` is indexed for semantic search, so an agent (or a CSM) can retrieve the exact ticket behind a tier. Swap the query for any theme -- outage, billing, cancellation.

In [ ]:
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
    'DEMO_C360_SEARCH',
    '{"query":"threatening to cancel the contract","columns":["CUSTOMER_ID","DOC_TYPE","CHUNK_TEXT"],"limit":5}'
  )
):results AS hits;

## 7 - Product health landscape -- the executive read

The corpus-intelligence head of the 360: a per-product rollup (customer count, average NPS, average error rate, high-risk count) with a 3-sentence executive briefing written by `AI_COMPLETE`. This is the slide a product leader wants -- grounded in the fused numbers, not a vibe.

In [ ]:
SELECT PRODUCT,
       CUSTOMER_COUNT,
       ROUND(AVG_NPS_Q2, 1)         AS avg_nps_q2,
       ROUND(AVG_MAX_ERROR_RATE, 3) AS avg_max_error_rate,
       HIGH_RISK_COUNT,
       EXEC_BRIEFING
FROM DT_DEMO_C360_HEALTH_LANDSCAPE
ORDER BY HIGH_RISK_COUNT DESC;

## 8 - Did it work? Computed tier vs. planted intent

The corpus is synthetic, so each customer was assigned a `COHORT_STORY` at synthesis (e.g. `error_plagued`, `silent_risk`, `vocal_churn`, `hidden_detractor`, `healthy`) that shaped all of their rows -- but the pipeline never keyed on it except as a guardrail. Do the computed risk tiers line up with what each cohort was designed to be? `error_plagued`, `silent_risk`, and `vocal_churn` should skew high; `billing_dispute`, `campaign_backlash`, and `hidden_detractor` should skew medium; `healthy` and `steady_growth` should skew low.

In [ ]:
SELECT cu.COHORT_STORY,
       COUNT(*)                         AS customers,
       COUNT_IF(r.RISK_TIER = 'high')   AS high,
       COUNT_IF(r.RISK_TIER = 'medium') AS medium,
       COUNT_IF(r.RISK_TIER = 'low')    AS low
FROM DEMO_C360_CUSTOMERS cu
JOIN DT_DEMO_C360_CUSTOMER_RECORD r ON r.CUSTOMER_ID = cu.CUSTOMER_ID
GROUP BY cu.COHORT_STORY
ORDER BY high DESC;

## Scale

This demo runs on **~40 customers** and a few dozen documents, but nothing about the pipeline is sized to that. It is `INCREMENTAL` end-to-end: landing a new document means staging it and loading its text into the file log, after which the **document-level** AI -- `AI_CLASSIFY` and the `AI_COMPLETE` sentiment call -- runs **once per document, ever** (unchanged docs are never reprocessed), and the structured `JOIN` re-runs cheaply on change. Two costs are *not* per-document and don't follow that rule: the per-product `AI_COMPLETE` briefing runs once per product on each health-landscape refresh, and the `DEMO_C360_SEARCH` service is a separate indexing + serving surface. So the document-level cost scales with *new documents*, not with the size of the book; the briefing and search costs scale with products and query volume.

> This demo builds over a **static corpus**: the sourcing script both stages the docs and backfills their `CONTENT`, and the ingest task stays suspended. For continuous ingestion, the task lands each new file's metadata, but its text still has to be loaded into `DEMO_C360_FILE_LOG.CONTENT` (the same step the sourcing script performs) before the sentiment and search steps can see it.

Adding a new signal is additive: a new structured table joins into the customer record, and a new document type is a new `AI_CLASSIFY` label plus a branch in the signal step -- no reprocessing of what's already landed.